In [120]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1337)

print(torch.__version__)

2.12.0+cpu


In [121]:
text = """
<USER>你好
<ASSISTANT>你好呀！
<END>
<USER>你叫什么名字？
<ASSISTANT>我是一个迷你聊天模型。
<END>
<USER>你会做什么？
<ASSISTANT>我可以尝试回答你的问题。
<END>
<USER>什么是人工智能？
<ASSISTANT>人工智能是让计算机完成一些通常需要人类智能的任务。
<END>
<USER>什么是Python？
<ASSISTANT>Python是一种编程语言。
<END>
<USER>我今天很开心。
<ASSISTANT>听起来不错！是什么让你这么开心？
<END>
<USER>我有点累。
<ASSISTANT>那就休息一会儿吧。
<END>
<USER>再见。
<ASSISTANT>再见，祝你今天愉快！
<END>
"""

# 重新建立字符词表
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join([itos[i] for i in ids])

data = torch.tensor(encode(text), dtype=torch.long)

print("文本字符数：", len(text))
print("词表大小：", vocab_size)
print("数据形状：", data.shape)

文本字符数： 350
词表大小： 97
数据形状： torch.Size([350])


In [122]:
batch_size = 4
block_size = 64

n_embd = 32
n_head = 4
n_layer = 2
dropout = 0.0

train_data = data

In [123]:
def get_batch(data):
    ix = torch.randint(len(data) - block_size, (batch_size,))
    
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    
    return x, y

In [124]:
token_embedding_table = nn.Embedding(vocab_size, n_embd)
tok_emb = token_embedding_table(xb)

In [125]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer(
            'tril',
            torch.tril(torch.ones(block_size, block_size))
        )

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        
        scores = q @ k.transpose(-2, -1)
        scores = scores * (k.shape[-1] ** -0.5)

        scores = scores.masked_fill(
            self.tril[:T, :T] == 0,
            float('-inf')
        )

        weights = F.softmax(scores, dim=-1)

        out = weights @ v

        return out

In [126]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        
        self.heads = nn.ModuleList([
            Head(head_size) for _ in range(num_heads)
        ])

        self.proj = nn.Linear(num_heads * head_size, n_embd)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out

In [127]:
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()

        head_size = n_embd // n_head

        self.sa = MultiHeadAttention(n_head, head_size)

        self.ffwd = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
        )

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [128]:
class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(
            vocab_size, n_embd
        )

        self.position_embedding_table = nn.Embedding(
            block_size, n_embd
        )

        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head) for _ in range(n_layer)]
        )

        self.ln_f = nn.LayerNorm(n_embd)

        self.lm_head = nn.Linear(n_embd, vocab_size)
        
    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)

        pos_emb = self.position_embedding_table(
            torch.arange(T, device=idx.device)
        )

        x = tok_emb + pos_emb

        x = self.blocks(x)

        x = self.ln_f(x)

        logits = self.lm_head(x)

        loss = None

        if targets is not None:
            B, T, C = logits.shape

            logits = logits.view(B * T, C)
            targets = targets.view(B * T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    

In [129]:
model = GPTLanguageModel()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [130]:
model.train()

for step in range(1000):
    xb, yb = get_batch(train_data)

    logits, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 50 == 0:
        print(f"step {step}: loss = {loss.item():.4f}")

step 0: loss = 4.8271
step 50: loss = 3.2668
step 100: loss = 2.3777
step 150: loss = 1.6388
step 200: loss = 1.1967
step 250: loss = 1.0375
step 300: loss = 0.6711
step 350: loss = 0.5168
step 400: loss = 0.4422
step 450: loss = 0.3915
step 500: loss = 0.3088
step 550: loss = 0.2876
step 600: loss = 0.2119
step 650: loss = 0.2149
step 700: loss = 0.1481
step 750: loss = 0.1658
step 800: loss = 0.1346
step 850: loss = 0.1017
step 900: loss = 0.1187
step 950: loss = 0.0599


In [131]:
@torch.no_grad()
def generate_reply(model, prompt, max_new_tokens=200):
    model.eval()

    # 将输入的对话上下文编码成数字
    idx = torch.tensor(
        [encode(prompt)],
        dtype=torch.long
    )

    # 逐个生成新字符
    for _ in range(max_new_tokens):

        # 模型最多只能看到最近 block_size 个字符
        idx_cond = idx[:, -block_size:]

        # 获取模型预测
        logits, _ = model(idx_cond)

        # 只关注最后一个位置的预测
        logits = logits[:, -1, :]

        # 将分数转换为概率
        probs = F.softmax(logits, dim=-1)

        # 抽取下一个字符
        idx_next = torch.multinomial(probs, num_samples=1)

        # 拼接到已有文本后面
        idx = torch.cat((idx, idx_next), dim=1)

        # 解码当前文本，检查是否生成了结束标记
        result = decode(idx[0].tolist())

        if "<END>" in result[len(prompt):]:
            break

    # 只返回模型新生成的内容
    result = decode(idx[0].tolist())
    return result[len(prompt):]

In [138]:
print("问题：你好")
print("回答：", generate_reply(
    model,
    "<USER>你好\n<ASSISTANT>"
))

print()

print("问题：什么是Python？")
print("回答：", generate_reply(
    model,
    "<USER>什么是Python？\n<ASSISTANT>"
))

问题：你好
回答： 我是一个迷你聊天模型。
<END>

问题：什么是Python？
回答： 人工智能是一种编程语言。
<END>
